# Z2005 — Week 1: Arrays and Dynamic Arrays
A self-study notebook covering static-array address arithmetic, how Python's `list` grows underneath `append`, a from-scratch `DynamicArray`, the true cost of insert/delete at different positions, and 2D arrays.

## Learning Objectives

By the end of this notebook you will be able to:

- Compute the memory address of `arr[i]` from a base address and element size, and explain why that makes indexing O(1).
- Explain, step by step, what happens when a dynamic array's `append` exceeds its capacity, and why doubling capacity keeps total copying at O(n) instead of O(n²).
- Implement a `DynamicArray` class from scratch, including manual resizing.
- State and justify the cost of insert/delete at the end, middle, and start of an array.
- Measure, with `timeit`, the real difference between appending and inserting at index 0 on a Python `list`.
- Compute addresses in a row-major 2D array.

## How to use this notebook

Run the cells top to bottom. `# TODO` cells are for you; `assert`-based cells are self-checks that fail loudly if your code is wrong and print a success message if it is right. Solutions are at the end.

## 1. Address arithmetic: why `arr[i]` is O(1)

Picture memory as a strip of numbered boxes. An array of same-size elements sits in one unbroken (**contiguous**) block: element 0 at the base address, element 1 immediately after it, and so on. Finding element `i` is never a search — it is one multiplication and one addition:

$$\text{address}(i) = \text{base\_address} + i \times \text{element\_size}$$

This is *why* `arr[i]` is O(1): the cost is the same arithmetic regardless of how large the array is or which index you ask for. A list of `Point` objects still indexes in O(1): Python stores fixed-size *references* (pointers) contiguously, and `points[i]` is one hop to the reference, then one hop to the actual object — two constant-time steps, still O(1) overall.

**The wall every plain array hits:** the memory immediately after an array may already belong to something else. Growing it in place would mean overwriting someone else's data, so a plain, fixed-size array can never safely grow — the only option is to allocate a brand-new, bigger block elsewhere and copy everything across. That is exactly the problem dynamic arrays (Section 2) solve.

In [ ]:
def address(base, i, element_size):
    # one multiplication, one addition -- constant work regardless of array size
    return base + i * element_size


# Worked examples from lecture: 4-byte ints, base address 500
assert address(500, 0, 4) == 500
assert address(500, 1, 4) == 504
assert address(500, 5, 4) == 520
# 8-byte doubles, base address 2000
assert address(2000, 10, 8) == 2080
print("1D address arithmetic checks passed")

### 2D arrays are one flat block wearing a disguise

`grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]` looks like rows and columns, but memory has no concept of rows — only one long line of addresses. **Row-major** layout stores row 0 completely, then row 1, then row 2: flattened, this is just `1, 2, 3, 4, 5, 6, 7, 8, 9`. The address formula adds a term that skips past every complete row before the row we want:

$$\text{address}(r, c) = \text{base} + (r \times \text{num\_cols} + c) \times \text{element\_size}$$

In [ ]:
def address_2d(base, row, col, num_cols, element_size):
    # r * num_cols skips every COMPLETE row before row r; + col then moves across within that row
    return base + (row * num_cols + col) * element_size


# 3x3 int grid, base 8000, num_cols=3: grid[2][1] should be at 8000 + (2*3+1)*4 = 8028
assert address_2d(8000, 2, 1, 3, 4) == 8028
# 4x5 int grid, base 0, num_cols=5: grid[3][2] should be at (3*5+2)*4 = 68
assert address_2d(0, 3, 2, 5, 4) == 68
print("2D row-major address arithmetic checks passed")

## 2. `DynamicArray` from scratch

A Python `list` behaves as though it can grow forever, one `append` at a time -- but Section 1 showed a fixed block cannot grow in place. Here is what actually happens underneath, simulated by hand: each time the backing storage is full, allocate a **new**, larger block; copy every existing element across; discard the old block; *then* insert the new element.

The critical design decision is *how much bigger* the new block should be. **Doubling** the capacity on every resize keeps the total copying work across `n` appends at O(n): resizes happen only `log2(n)` times, and the total elements ever copied is bounded by `n + n/2 + n/4 + ... < 2n`. Growing by a fixed amount (e.g. "add one slot") instead would trigger a full copy on almost every single append, giving `1 + 2 + 3 + ... + n ≈ n²/2` total copies — O(n²), dramatically worse at scale (roughly 2 million copies for doubling at n = 1,000,000, versus roughly 500 billion for grow-by-1).

Below is a fully working reference `DynamicArray`; Exercise 2 asks you to build one of your own from a partial skeleton.

In [ ]:
class DynamicArray:
    def __init__(self):
        self.capacity = 1
        self.length = 0
        self._backing = [None] * self.capacity   # the fixed-size block acting as "raw memory"

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        if i < 0 or i >= self.length:
            raise IndexError(f"index {i} out of range for length {self.length}")
        return self._backing[i]

    def _resize(self, new_capacity):
        new_backing = [None] * new_capacity        # allocate a brand-new, bigger block
        for i in range(self.length):
            new_backing[i] = self._backing[i]       # copy every existing element across
        self._backing = new_backing                 # old block is discarded (garbage collected)
        self.capacity = new_capacity

    def append(self, x):
        if self.length == self.capacity:
            self._resize(self.capacity * 2)         # doubling keeps total copying at O(n)
        self._backing[self.length] = x
        self.length += 1


arr = DynamicArray()
for i in range(20):
    arr.append(i)

assert len(arr) == 20
assert arr[0] == 0 and arr[19] == 19
assert arr.capacity == 32   # doubling sequence: 1, 2, 4, 8, 16, 32
print("DynamicArray append + resize checks passed, final capacity:", arr.capacity)

## 3. Length vs. capacity, and measuring append vs. front-insert

`len(arr)` counts elements actually stored; `capacity` counts slots reserved. Most appends increase length without changing capacity at all — they simply use a slot that was already free. We can watch Python's own `list` do this via `sys.getsizeof`, which jumps only when a resize actually happens.

In [ ]:
import sys

scores = []
prev_size = None
for i in range(10):
    scores.append(i)
    size = sys.getsizeof(scores)
    resized = "  <- capacity grew" if size != prev_size else ""
    print(f"len={len(scores):>2}  sys.getsizeof={size:>4}{resized}")
    prev_size = size

The theory above predicts that appending at the end of a list stays cheap (amortized O(1)) while inserting at the front (`insert(0, x)`) is O(n) per call, because every existing element must shift right to make room. We do not assume specific timing numbers — we measure them live with `timeit`, and expect the *ratio* between the two to grow as `n` grows.

In [ ]:
import timeit

def time_append(n):
    data = []
    def run():
        for i in range(n):
            data.append(i)
    return timeit.timeit(run, number=1)

def time_front_insert(n):
    data = []
    def run():
        for i in range(n):
            data.insert(0, i)
    return timeit.timeit(run, number=1)

for n in (2_000, 4_000, 8_000):
    t_append = time_append(n)
    t_front = time_front_insert(n)
    ratio = t_front / t_append if t_append > 0 else float("inf")
    print(f"n={n:>6}  append={t_append:.4f}s  insert(0)={t_front:.4f}s  ratio={ratio:.1f}x")

Watch the ratio column: it should grow as `n` doubles. `append` stays close to linear in `n` overall (amortized O(1) per call); `insert(0, ...)` grows closer to quadratic in `n` overall (O(n) per call, from the shifting argument below), so the *total* cost of `n` front-inserts is O(n²).

## 4. Insert/delete: where the real cost hides

Appending or popping at the *end* of an array touches no existing element — O(1) (amortized, ignoring the occasional resize). Inserting or deleting in the *middle* or at the *start* is different: making room (or closing a gap) requires physically shifting every element after the target position.

Inserting at index `k` into an array of `n` elements shifts `n - k` elements: shifting happens right-to-left (moving the last element first) so nothing is overwritten before it is copied. Inserting near the end is cheap (few shifts); inserting at index 0 is the worst case — **all** `n` elements shift.

| Operation | At the end | In the middle | At the start |
|---|---|---|---|
| Insert | O(1) amortized | O(n) | O(n) |
| Delete | O(1) | O(n) | O(n) |
| Access by index | O(1) | O(1) | O(1) |

This connects directly back to Week 0's binary search: keeping an array *sorted* lets you **find** the correct insertion position in O(log n) via binary search, but **making room** for it still costs O(n) via shifting. The net cost of inserting into a sorted array is O(n), dominated by the shift, not the search.

In [ ]:
def find_sorted_position(arr, x):
    # binary search for the correct insertion index -- O(log n)
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def insert_sorted(arr, x):
    pos = find_sorted_position(arr, x)   # O(log n): find WHERE
    arr.insert(pos, x)                   # O(n): MAKE ROOM by shifting
    return arr


leaderboard = [45, 60, 71, 88, 92, 99]
insert_sorted(leaderboard, 80)
assert leaderboard == [45, 60, 71, 80, 88, 92, 99]
print("Sorted insertion matches the worked trace:", leaderboard)

**Why `collections.deque` exists.** A `deque` is implemented as a doubly linked list of fixed-size blocks, not one contiguous array. It trades away O(1) random index access (`deque` indexing is O(n)) for genuinely O(1) insertion and removal at *both* ends — no shifting at all, because nothing physically moves. Use `list` when you mostly index and append; use `deque` when you frequently push or pop from the front.

In [ ]:
from collections import deque

d = deque([67, 88, 45, 92, 71])
d.appendleft(99)   # O(1) -- NOT O(n) like list.insert(0, ...)
assert list(d) == [99, 67, 88, 45, 92, 71]
print("deque.appendleft is O(1):", list(d))

## Exercises

### Exercise 1 — address arithmetic

Implement `address(base, i, element_size)` and `address_2d(base, row, col, num_cols, element_size)` (you already saw the working versions above in Section 1 -- reimplement them here from the formula, without looking back, as a self-test).

Example: `address(1000, 3, 4)` should be `1012`; `address_2d(0, 2, 1, 4, 4)` should be `36`.

In [ ]:
def ex_address(base, i, element_size):
    # TODO: return base + i * element_size
    raise NotImplementedError

def ex_address_2d(base, row, col, num_cols, element_size):
    # TODO: return base + (row * num_cols + col) * element_size
    raise NotImplementedError

**Self-check — Exercise 1**

In [ ]:
assert ex_address(1000, 3, 4) == 1012
assert ex_address(500, 0, 4) == 500
assert ex_address_2d(0, 2, 1, 4, 4) == 36
assert ex_address_2d(8000, 2, 1, 3, 4) == 8028
print("\u2705 Exercise 1 passed")

### Exercise 2 — build your own `DynamicArray`

Complete `MyDynamicArray` below: start at capacity 1, double capacity whenever `append` would overflow it, and support `len()` and indexing. This mirrors the reference implementation from Section 2 -- write it without copying that cell verbatim.

In [ ]:
class MyDynamicArray:
    def __init__(self):
        self.capacity = 1
        self.length = 0
        self._backing = [None] * self.capacity

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        # TODO: raise IndexError if i is out of range [0, length), else return self._backing[i]
        raise NotImplementedError

    def _resize(self, new_capacity):
        # TODO: allocate a new backing list of new_capacity Nones, copy self.length
        # elements across, then replace self._backing and update self.capacity
        raise NotImplementedError

    def append(self, x):
        # TODO: resize (doubling) if full, then store x and increment length
        raise NotImplementedError

**Self-check — Exercise 2**

In [ ]:
a = MyDynamicArray()
for i in range(10):
    a.append(i * i)
assert len(a) == 10
assert [a[i] for i in range(10)] == [i * i for i in range(10)]
assert a.capacity == 16   # doubling: 1, 2, 4, 8, 16
try:
    a[10]
    raise AssertionError("should raise IndexError past the end")
except IndexError:
    pass
print("\u2705 Exercise 2 passed")

### Exercise 3 — `insert_sorted` and timing it

Implement `my_insert_sorted(arr, x)`: use binary search to find the correct position in an already-sorted list `arr`, then insert `x` there (in place), and return `arr`.

In [ ]:
def my_insert_sorted(arr, x):
    # TODO: binary search for the correct position, then arr.insert(pos, x); return arr
    raise NotImplementedError

**Self-check — Exercise 3**

In [ ]:
data = [10, 20, 30, 50, 60]
my_insert_sorted(data, 40)
assert data == [10, 20, 30, 40, 50, 60]
my_insert_sorted(data, 5)
assert data == [5, 10, 20, 30, 40, 50, 60]
print("\u2705 Exercise 3 passed")

### Exercise 4 (harder) — a `Matrix` class over one flat list

Implement `Matrix`, storing all elements in a single flat Python list (`self._data`), row-major, exactly like Section 1's 2D address arithmetic (but here `element_size` is implicit -- we are indexing into a Python list, not computing byte addresses). Support `__init__(self, rows, cols, fill=0)`, `get(self, r, c)`, and `set(self, r, c, value)`.

Example:
```python
m = Matrix(2, 3)
m.set(1, 2, 99)
m.get(1, 2)   # 99
```

In [ ]:
class Matrix:
    def __init__(self, rows, cols, fill=0):
        self.rows = rows
        self.cols = cols
        # TODO: store rows * cols copies of `fill` in self._data (one flat list)
        raise NotImplementedError

    def _index(self, r, c):
        # TODO: row-major flat index: r * self.cols + c
        raise NotImplementedError

    def get(self, r, c):
        # TODO: return self._data[self._index(r, c)]
        raise NotImplementedError

    def set(self, r, c, value):
        # TODO: self._data[self._index(r, c)] = value
        raise NotImplementedError

**Self-check — Exercise 4**

In [ ]:
m = Matrix(2, 3, fill=0)
assert m.get(0, 0) == 0
m.set(1, 2, 99)
assert m.get(1, 2) == 99
assert m.get(0, 1) == 0   # untouched cells remain at fill value
assert len(m._data) == 6
print("\u2705 Exercise 4 passed")

## Quiz

**1. Why is `arr[i]` O(1) on a plain array, but a linked-list `node[i]` is not (a preview of Week 4)?**
<details><summary>Show answer</summary>An array's address formula (<code>base + i * element_size</code>) computes any element's location directly with one multiplication and addition -- arithmetic, not search. A linked list has no such formula: reaching element <code>i</code> means following <code>i</code> pointers one at a time from the head, which is O(i).</details>

**2. Why does doubling capacity keep total copying at O(n) over n appends, while growing by a fixed amount does not?**
<details><summary>Show answer</summary>Doubling means resizes happen only <code>log2(n)</code> times, and the total elements ever copied across all resizes is bounded by <code>n + n/2 + n/4 + ... &lt; 2n</code> -- O(n). Growing by a fixed small amount instead triggers a full copy on almost every append, giving roughly <code>n^2/2</code> total copies -- O(n^2).</details>

**3. For an array of 1,000 elements, rank `arr.append(x)`, `arr.insert(500, x)`, and `arr.insert(0, x)` from cheapest to most expensive.**
<details><summary>Show answer</summary><code>arr.append(x)</code> (O(1) amortized, no shifting) &lt; <code>arr.insert(500, x)</code> (O(n), shifts roughly 500 elements) &lt; <code>arr.insert(0, x)</code> (O(n), shifts all 1,000 elements -- the most expensive of the three).</details>

**4. What trade-off does `collections.deque` make compared to `list`?**
<details><summary>Show answer</summary><code>deque</code> gives O(1) insertion/removal at BOTH ends (no shifting, since it is a doubly linked list of blocks), at the cost of O(n) random index access instead of list's O(1).</details>

## Solutions (try the exercises yourself first!)

In [ ]:
# --- Exercise 1 solution ---
def ex_address(base, i, element_size):
    return base + i * element_size

def ex_address_2d(base, row, col, num_cols, element_size):
    return base + (row * num_cols + col) * element_size

assert ex_address(1000, 3, 4) == 1012
assert ex_address_2d(0, 2, 1, 4, 4) == 36
print("Exercise 1 solution verified")

In [ ]:
# --- Exercise 2 solution ---
class MyDynamicArray:
    def __init__(self):
        self.capacity = 1
        self.length = 0
        self._backing = [None] * self.capacity

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        if i < 0 or i >= self.length:
            raise IndexError(f"index {i} out of range for length {self.length}")
        return self._backing[i]

    def _resize(self, new_capacity):
        new_backing = [None] * new_capacity
        for i in range(self.length):
            new_backing[i] = self._backing[i]
        self._backing = new_backing
        self.capacity = new_capacity

    def append(self, x):
        if self.length == self.capacity:
            self._resize(self.capacity * 2)
        self._backing[self.length] = x
        self.length += 1


a = MyDynamicArray()
for i in range(10):
    a.append(i * i)
assert a.capacity == 16
print("Exercise 2 solution verified")

In [ ]:
# --- Exercise 3 solution ---
def my_insert_sorted(arr, x):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    arr.insert(lo, x)
    return arr


data = [10, 20, 30, 50, 60]
my_insert_sorted(data, 40)
assert data == [10, 20, 30, 40, 50, 60]
print("Exercise 3 solution verified")

In [ ]:
# --- Exercise 4 solution ---
class Matrix:
    def __init__(self, rows, cols, fill=0):
        self.rows = rows
        self.cols = cols
        self._data = [fill] * (rows * cols)

    def _index(self, r, c):
        return r * self.cols + c

    def get(self, r, c):
        return self._data[self._index(r, c)]

    def set(self, r, c, value):
        self._data[self._index(r, c)] = value


m = Matrix(2, 3)
m.set(1, 2, 99)
assert m.get(1, 2) == 99
print("Exercise 4 solution verified")

## MTech Extension — amortized analysis: the accounting-method proof

"`append` is amortized O(1)" and "some individual `append` calls trigger an O(n) copy" are both true at once -- the apparent contradiction is resolved by **amortized analysis**. The accounting (coin-jar) method: charge every append a flat fee of 3 "coins" -- 1 spent immediately on inserting the element, 2 banked in a savings jar untouched.

When a resize doubles capacity from `k` to `2k`, exactly `k` elements must be copied. Each of those `k` elements *already* banked 2 coins when it was first inserted -- `k` elements x 2 coins = `2k` coins available, exactly enough to pay for copying all `k` of them (with coins to spare, since inserting the new element costs only 1 more). The jar balance never goes negative. Since every append deposits a flat 3 coins, `n` appends cost at most `3n` coins total -- O(n) total, hence O(1) *average* per append. This is what "amortized" means: not that every call is fast, but that the average over any long run is O(1).

The rigorous version replaces the coin jar with a **potential function** $\Phi(\text{array}) = 2 \cdot \text{length} - \text{capacity}$, which must stay $\ge 0$ throughout. The amortized cost of an operation is defined as (actual cost) + (change in $\Phi$):

- **No resize:** actual cost 1 (just the insert); length increases by 1, capacity unchanged, so $\Delta\Phi = 2$. Amortized cost $= 1 + 2 = 3$.
- **Resize (doubling):** actual cost $k+1$ (copy $k$ elements, insert 1 new one); capacity doubles from $k$ to $2k$ while length goes from $k$ to $k+1$, so $\Delta\Phi = 2(k+1) - 2k - (2k - k) = 2 - k$. Amortized cost $= (k+1) + (2-k) = 3$.

Both cases give the same constant, 3 -- that equality across both the cheap and expensive case is the entire proof. The cell below verifies this arithmetic directly by simulating a doubling dynamic array and tracking $\Phi$ after every append.

In [ ]:
def simulate_amortized_cost(n_appends):
    """Track length, capacity, actual cost, and potential Phi = 2*length - capacity
    after every append, and verify the amortized cost (actual + delta-Phi) is always 3.
    """
    length, capacity = 0, 1
    phi_prev = 2 * length - capacity   # Phi(0) = 2*0 - 1 = -1, the true starting potential
    amortized_costs = []
    for _ in range(n_appends):
        if length == capacity:
            # resize: actual cost is copying `length` elements, plus inserting 1
            actual_cost = length + 1
            capacity *= 2
        else:
            actual_cost = 1
        length += 1
        phi_now = 2 * length - capacity
        delta_phi = phi_now - phi_prev
        amortized_costs.append(actual_cost + delta_phi)
        phi_prev = phi_now
    return amortized_costs


costs = simulate_amortized_cost(20)
print("amortized cost per append:", costs)
assert all(c == 3 for c in costs), "every amortized cost should equal the constant 3"
print("Confirmed: every one of the 20 appends has amortized cost exactly 3, "
      "regardless of whether that specific append triggered a resize.")

**Discussion point for MTech:** the same accounting method explains why a stack's `pop()` on a dynamic array can also shrink capacity when the array becomes very empty, *without* thrashing (repeatedly resizing back and forth), as long as it only shrinks once the array drops to 1/4 full rather than 1/2 full. Sketch, in the coin-jar style, why 1/2 as the shrink threshold would thrash on an alternating append/pop/append/pop sequence, while 1/4 does not.